In [1]:
import requests
import datetime



In [2]:
# Prognosedatenabruf über Open-Meteo-API

def fetch_forecast_data(latitude, longitude, hourly_params, timezone, start_date, end_date, tilt = 0, azimuth = 0):
    """
    Ruft die Vorhersagedaten von Open-Meteo für die angegebenen Parameter ab.
    """
    url = (
        f"https://api.open-meteo.com/v1/forecast?"
        f"latitude={latitude}&longitude={longitude}"
        f"&hourly={hourly_params}"
        f"&start_date={start_date}&end_date={end_date}"
        f"&timezone={timezone}"
        f"&tilt={tilt}&azimuth={azimuth}"
    )
    try:
        response = requests.get(url)
        response.raise_for_status()  # Löst einen Fehler aus, wenn der HTTP-Statuscode nicht 200 ist.
        return response.json()
    except requests.RequestException as e:
        print("Fehler beim Abrufen der Daten:", e)
        return None







In [6]:
 # Beispielkoordinaten: Schleiz
latitude = 50.5795188171788
longitude = 11.81734624374329

# Gewünschte stündliche Parameter:
# - temperature_2m: Temperatur in 2 m Höhe in °C
# - shortwave_radiation: Solare Einstrahlung auf eine horizontale Fläche (W/m²)
# - cloudcover: Bewölkungsgrad in %
hourly_params = "temperature_2m,shortwave_radiation,global_tilted_irradiance,cloudcover"

# Zeitzone (z. B. "Europe/Berlin")
timezone = "Europe/Berlin"

# Bestimme das aktuelle Datum und das Datum des nächsten Tages im Format YYYY-MM-DD
today = datetime.date.today()
previewDay = today + datetime.timedelta(days=0)
start_date = previewDay.strftime("%Y-%m-%d")
end_date = previewDay.strftime("%Y-%m-%d")
# Verkippung zur Horizontalen
tilt = 0;
# Verdrehung gegen Süd (0°) im Uhrzeigersinn
azimuth = 0;
#end_date = tomorrow.strftime("%Y-%m-%d")

print("Abrufen der Vorhersagedaten von Open-Meteo...")
data = fetch_forecast_data(latitude, longitude, hourly_params, timezone, start_date, end_date, tilt, azimuth)

if data:
    # Lese die stündlichen Vorhersagedaten aus
    hourly = data.get("hourly", {})
    times = hourly.get("time", [])
    temperatures = hourly.get("temperature_2m", [])
    radiation = hourly.get("shortwave_radiation", [])
    radiationTilt = hourly.get("global_tilted_irradiance", [])
    cloudcover = hourly.get("cloudcover", [])

    # Strahlung auf PV-Leistung 6,6kWp umrechnen
    # Fläche 29,3m^2
    A = 29.3
    # Wirkungsgrad 22,3%
    etha = 0.223
    # Leistung in kW
    P = [x * A * etha / 1000 for x in radiation]
    PTilt = [x * A * etha / 1000 for x in radiationTilt]
    # Gesamtleistung pro Tag in kWh
    power = sum(P)
    powerTilt = sum(PTilt)
    print(f"\nGesamte PV-Leistung pro Tag: {power} kWh")
    print(f"Gesamte PV-Leistung pro Tag (verkipppt): {powerTilt} kWh")
    print("\nStündliche Vorhersage:")
    for i in range(len(times)):
        print(f"{times[i]} - Temperatur: {temperatures[i]}°C, "
                f"Solare Einstrahlung: {radiation[i]} W/m², "
                f"Bewölkungsgrad: {cloudcover[i]}%, "
                f"PV-Leistung: {P[i]} kW")
    
    
    #for i in range(len(radiation)):
       
else:
    print("Keine Daten erhalten.")

Abrufen der Vorhersagedaten von Open-Meteo...

Gesamte PV-Leistung pro Tag: 16.71567637 kWh
Gesamte PV-Leistung pro Tag (verkipppt): 16.71502298 kWh

Stündliche Vorhersage:
2026-03-20T00:00 - Temperatur: 3.8°C, Solare Einstrahlung: 0.0 W/m², Bewölkungsgrad: 2%, PV-Leistung: 0.0 kW
2026-03-20T01:00 - Temperatur: 1.9°C, Solare Einstrahlung: 0.0 W/m², Bewölkungsgrad: 30%, PV-Leistung: 0.0 kW
2026-03-20T02:00 - Temperatur: 0.9°C, Solare Einstrahlung: 0.0 W/m², Bewölkungsgrad: 28%, PV-Leistung: 0.0 kW
2026-03-20T03:00 - Temperatur: -0.0°C, Solare Einstrahlung: 0.0 W/m², Bewölkungsgrad: 98%, PV-Leistung: 0.0 kW
2026-03-20T04:00 - Temperatur: 0.7°C, Solare Einstrahlung: 0.0 W/m², Bewölkungsgrad: 87%, PV-Leistung: 0.0 kW
2026-03-20T05:00 - Temperatur: 0.3°C, Solare Einstrahlung: 0.0 W/m², Bewölkungsgrad: 78%, PV-Leistung: 0.0 kW
2026-03-20T06:00 - Temperatur: -0.2°C, Solare Einstrahlung: 0.0 W/m², Bewölkungsgrad: 99%, PV-Leistung: 0.0 kW
2026-03-20T07:00 - Temperatur: -0.1°C, Solare Einstrahlu

In [26]:
print(radiation)

[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 5.0, 90.2, 238.5, 386.0, 501.0, 563.2, 566.8, 499.5, 455.5, 350.5, 199.0, 53.2, 0.5, 0.0, 0.0, 0.0, 0.0]


In [27]:
print(cloudcover)

[0, 0, 0, 0, 0, 0, 0, 0, 4, 0, 0, 9, 18, 100, 100, 0, 22, 100, 94, 69, 100, 100, 100, 100]


In [28]:
print(start_date)
print(end_date)

2026-03-13
2026-03-13
